# MindBridge: DistilBERT sentiment training
Train a **separate sentiment classifier**. Your UlizaLlama3 notebook continues to train replies.

This notebook prepares a three-class model: negative, neutral and positive. Sentiment does not diagnose mental illness or measure suicide probability. Keep `SENTIMENT_RISK_ENABLED=0` until project evaluation and review are complete.

The starting encoder is English DistilBERT. Including Kiswahili and Sheng training examples does not establish that it works well in those languages; evaluate them separately.

**Current status:** workflow prepared; no project-labelled data supplied, no fine-tuned model or evaluation scores claimed.


## 1. Prepare the reviewed CSV first
Use participant/user messages only, not counsellor/assistant replies. Keep identifying information out. Record source and permitted use before reusing dataset text.

Required columns: `sample_id,group_id,text,language,source,source_license,split,rater_1,rater_2,adjudicator`.

- `language`: english, kiswahili, sheng or code_switched.
- Each rater works independently: negative = expressed distress/unpleasant feelings; positive = expressed pleasant feelings/support; neutral = no clear positive or negative sentiment. These are sentiment labels, not risk levels. Flag ambiguous/context-dependent examples for review; do not guess from keywords or mood scores.
- If raters agree, leave adjudicator blank. If they disagree, a third reviewer supplies the final label.
- `group_id` links all turns of a conversation, duplicates, translations and rewritten variants to the same original. Complete these links from your dataset manifest; the notebook cannot infer translation lineage.
- `split` is train, validation or test. Split original groups before translating or augmenting. Preserve previously held-out groups. Never reuse generation-model training conversations as an independent end-to-end system test.
- Aim for meaningful class coverage in each language and enough held-out groups for useful uncertainty estimates. Small test samples cannot establish deployment readiness.

The companion CSV is an empty template. Do not train on the two demo sentences, automatically generated labels or model predictions presented as human-reviewed labels.


## 2. Kaggle or Colab setup
Create a **new notebook** so this does not interrupt the UlizaLlama3 training session. Enable an available GPU and Internet. Attach the labelled CSV as a private input. No Hugging Face token is needed for the public encoder. Run cells in order.


In [ ]:
%pip install -q "transformers==4.57.6" "datasets==4.3.0" "accelerate==1.11.0" "scikit-learn==1.7.2" "pandas>=2.2,<3"


In [ ]:
import os, json, hashlib, platform, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, cohen_kappa_score
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)

reviewed_inputs = sorted(Path("/kaggle/input").rglob("sentiment_labels.csv"))
if len(reviewed_inputs) != 1:
    raise RuntimeError("Attach exactly one reviewed sentiment_labels.csv as a private Kaggle input. Training has not started.")
DATA_PATH = reviewed_inputs[0]
OUTPUT = Path("/kaggle/working/mindbridge_sentiment")  # Colab: /content/mindbridge_sentiment
BASE_MODEL = "distilbert/distilbert-base-uncased"
BASE_REVISION = "12040accade4e8a0f71eabdb258fecc2e7e948be"
LABELS = ["negative", "neutral", "positive"]
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
LANGUAGES = {"english", "kiswahili", "sheng", "code_switched"}
SEED = 42
MAX_LENGTH = 512  # same truncation limit as the Django adapter
THRESHOLD = 0.85  # decide using validation only, then freeze before test evaluation
set_seed(SEED)
OUTPUT.mkdir(parents=True, exist_ok=True)
print("GPU available:", torch.cuda.is_available())
print("The next cell stops unless the reviewed CSV is ready.")


In [ ]:
def validate_labels(frame):
    required = ["sample_id", "group_id", "text", "language", "source", "source_license",
                "split", "rater_1", "rater_2", "adjudicator"]
    missing = set(required) - set(frame.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    df = frame.copy().fillna("")
    for col in required:
        df[col] = df[col].astype(str).str.strip()
    if df.empty:
        raise ValueError("No labelled examples yet. Complete the template first.")
    for col in required[:-1]:
        if df[col].eq("").any():
            raise ValueError(f"Blank values in {col}; review the dataset first.")
    if df.sample_id.duplicated().any():
        raise ValueError("sample_id must be unique.")
    for col in ["rater_1", "rater_2"]:
        if not df[col].isin(LABELS).all():
            raise ValueError(f"Invalid sentiment label in {col}.")
    if not df.adjudicator.isin([""] + LABELS).all():
        raise ValueError("Invalid adjudicator label.")
    disagree = df.rater_1.ne(df.rater_2)
    if (disagree & df.adjudicator.eq("")).any():
        raise ValueError("Every disagreement needs a third-reviewer label.")
    if ((~disagree) & df.adjudicator.ne("")).any():
        raise ValueError("Leave adjudicator blank when both raters agree.")
    if not df.language.isin(LANGUAGES).all():
        raise ValueError("Use the four documented language categories.")
    if not df.split.isin(["train", "validation", "test"]).all():
        raise ValueError("split must be train, validation or test.")
    if df.groupby("group_id")["split"].nunique().gt(1).any():
        raise ValueError("Leakage: an original group appears in several splits.")
    canonical = df.text.str.casefold().str.replace(r"\s+", " ", regex=True)
    df["text_hash"] = canonical.map(lambda text: hashlib.sha256(text.encode()).hexdigest())
    if df.groupby("text_hash")["split"].nunique().gt(1).any():
        raise ValueError("Leakage: duplicate text appears in several splits.")
    df["label"] = np.where(disagree, df.adjudicator, df.rater_1)
    if df.groupby("text_hash")["label"].nunique().gt(1).any():
        raise ValueError("Identical text has conflicting final labels; review it.")
    df["labels"] = df.label.map(LABEL_TO_ID)
    for split in ["train", "validation", "test"]:
        subset = df[df.split.eq(split)]
        if set(subset.label) != set(LABELS):
            raise ValueError(f"{split} must include all three sentiment classes.")
    return df

if not DATA_PATH.is_file():
    raise FileNotFoundError("Attach the reviewed CSV and edit DATA_PATH above.")
df = validate_labels(pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False))
print(pd.crosstab([df.split, df.language], df.label))
print("Unique original groups:", df.groupby("split").group_id.nunique().to_dict())
print("Rater agreement:", float(df.rater_1.eq(df.rater_2).mean()))
kappa = cohen_kappa_score(df.rater_1, df.rater_2, labels=LABELS)
print("Cohen's kappa:", float(kappa) if np.isfinite(kappa) else "undefined")
# Keep this ledger private: IDs and text hashes can still disclose dataset membership.
df[["sample_id", "group_id", "split", "language", "label", "text_hash"]].to_csv(
    OUTPUT / "split_ledger.csv", index=False)
DATA_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()


## 3. Tokenization and training
The classifier head starts fresh with three outputs. Fine-tune the encoder and head on train only; select the best epoch using validation macro-F1. Do not compare test results to choose epochs or hyperparameters.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_REVISION, trust_remote_code=False)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, revision=BASE_REVISION, num_labels=3,
    id2label=dict(enumerate(LABELS)), label2id=LABEL_TO_ID,
    trust_remote_code=False, use_safetensors=True)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

parts = {}
for split in ["train", "validation", "test"]:
    subset = df[df.split.eq(split)].reset_index(drop=True)
    parts[split] = Dataset.from_pandas(subset[["text", "labels"]], preserve_index=False).map(
        tokenize, batched=True, remove_columns=["text"])
    raw_tokens = tokenizer(subset.text.tolist(), truncation=False)["input_ids"]
    print(split, "truncated messages:", sum(len(ids) > MAX_LENGTH for ids in raw_tokens))
    for language, group in subset.groupby("language"):
        token_ids = tokenizer(group.text.tolist(), truncation=True, max_length=MAX_LENGTH)["input_ids"]
        total = sum(len(ids) for ids in token_ids)
        unknown = sum(ids.count(tokenizer.unk_token_id) for ids in token_ids)
        print(split, language, "unknown-token fraction:", round(unknown / max(total, 1), 4))

def compute_metrics(pred):
    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    labels = pred.label_ids
    predicted = logits.argmax(axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, predicted, labels=[0,1,2],
                                                 average="macro", zero_division=0)
    return {"macro_precision": p, "macro_recall": r, "macro_f1": f1}

args = TrainingArguments(
    output_dir=str(OUTPUT / "checkpoints"), learning_rate=2e-5, num_train_epochs=3,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    weight_decay=0.01, eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
    save_total_limit=2, save_safetensors=True, seed=SEED, data_seed=SEED,
    report_to="none", push_to_hub=False, fp16=torch.cuda.is_available(),
    dataloader_num_workers=0)
trainer = Trainer(model=model, args=args, train_dataset=parts["train"],
                  eval_dataset=parts["validation"], processing_class=tokenizer,
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
trainer.train()


## 4. Validation threshold sweep
Evaluate negative sentiment at thresholds 0.80–0.90. A positive decision requires **negative to be the predicted class AND its probability to meet the threshold**, matching Django. A high threshold can reduce recall. This is sentiment precision/recall, not alert precision/recall; evaluate the hybrid engine separately.

Review validation support and errors before selecting `THRESHOLD`. Keep 0.85 provisional unless the data supports it. Freeze the choice before running the test cell. If no threshold is suitable, report that and keep sentiment risk contribution off.


In [ ]:
def probabilities(result):
    logits = result.predictions[0] if isinstance(result.predictions, tuple) else result.predictions
    return torch.softmax(torch.tensor(logits), dim=-1).numpy()

def threshold_metrics(y_true, probs, threshold):
    selected = (probs.argmax(axis=1) == 0) & (probs[:, 0] >= threshold)
    truth = np.asarray(y_true) == 0
    tp = int((selected & truth).sum())
    fp = int((selected & ~truth).sum())
    fn = int((~selected & truth).sum())
    precision = tp / (tp + fp) if tp + fp else None
    recall = tp / (tp + fn) if tp + fn else None
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else None
    return {"threshold": float(threshold), "precision": precision, "recall": recall,
            "f1": f1, "predicted_negative": tp + fp, "actual_negative": tp + fn,
            "false_positive": fp, "false_negative": fn}

val_result = trainer.predict(parts["validation"])
val_probs = probabilities(val_result)
val_frame = df[df.split.eq("validation")].reset_index(drop=True)
sweep = pd.DataFrame([threshold_metrics(val_frame.labels, val_probs, t)
                      for t in np.round(np.arange(0.80, 0.901, 0.01), 2)])
sweep.to_csv(OUTPUT / "validation_thresholds.csv", index=False)
display(sweep)
# Edit only after reviewing VALIDATION. Never choose this from test results.
THRESHOLD = 0.85


## 5. Frozen held-out evaluation
Run after freezing the training choices and threshold. Reports include class metrics, confusion matrices and negative-threshold metrics overall and per language. Missing language strata and small samples are explicit limitations.

95% intervals below resample **original groups**, rather than treating translated versions as independent. They describe this sample and do not establish clinical safety.


In [ ]:
def bootstrap_f1(frame, probs, draws=500):
    groups = frame.group_id.unique()
    if len(groups) < 2:
        return {"lower": None, "upper": None, "groups": len(groups)}
    rng = np.random.default_rng(SEED)
    indices = {g: np.flatnonzero(frame.group_id.to_numpy() == g) for g in groups}
    values = []
    truth = frame.labels.to_numpy()
    predicted = probs.argmax(axis=1)
    for _ in range(draws):
        sampled = rng.choice(groups, size=len(groups), replace=True)
        ix = np.concatenate([indices[g] for g in sampled])
        values.append(precision_recall_fscore_support(truth[ix], predicted[ix],
                      labels=[0,1,2], average="macro", zero_division=0)[2])
    low, high = np.quantile(values, [0.025, 0.975])
    return {"lower": float(low), "upper": float(high), "groups": len(groups)}

def evaluation_report(frame, probs):
    reports = {}
    for language in ["overall"] + sorted(LANGUAGES):
        ix = np.arange(len(frame)) if language == "overall" else np.flatnonzero(frame.language.eq(language))
        if not len(ix):
            reports[language] = {"status": "missing stratum", "samples": 0}
            continue
        subset = frame.iloc[ix].reset_index(drop=True)
        selected_probs = probs[ix]
        reports[language] = {
            "samples": len(ix), "groups": int(subset.group_id.nunique()),
            "class_counts": subset.label.value_counts().to_dict(),
            "classification": classification_report(subset.labels, selected_probs.argmax(axis=1),
                 labels=[0,1,2], target_names=LABELS, output_dict=True, zero_division=0),
            "confusion_matrix": confusion_matrix(subset.labels, selected_probs.argmax(axis=1),
                                                  labels=[0,1,2]).tolist(),
            "negative_threshold": threshold_metrics(subset.labels, selected_probs, THRESHOLD),
            "macro_f1_95pct_group_bootstrap": bootstrap_f1(subset, selected_probs)}
    return reports

if not 0.80 <= THRESHOLD <= 0.90:
    raise ValueError("Freeze a validation-reviewed threshold in the documented 0.80–0.90 range.")
test_frame = df[df.split.eq("test")].reset_index(drop=True)
test_probs = probabilities(trainer.predict(parts["test"]))
report = evaluation_report(test_frame, test_probs)
(OUTPUT / "test_metrics.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))
predictions = test_frame[["sample_id", "group_id", "language", "label"]].copy()
predictions["predicted_label"] = [LABELS[i] for i in test_probs.argmax(axis=1)]
for i, label in enumerate(LABELS):
    predictions[f"p_{label}"] = test_probs[:, i]
predictions.to_csv(OUTPUT / "test_predictions_private.csv", index=False)
print("Review errors against the private input CSV; do not publish participant text.")


## 6. Baseline comparison on the same held-out examples
Compare with the currently installed SST-2 baseline. It has only negative and positive labels; it cannot predict neutral. Report this capability difference when comparing three-class metrics.


In [ ]:
BASELINE_MODEL = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
BASELINE_REVISION = "714eb0fa89d2f80546fda750413ed43d93601a13"
# Save the trained model before moving it off GPU for baseline evaluation.
MODEL_DIR = OUTPUT / "model"
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(MODEL_DIR)
trainer.model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()
baseline_tok = AutoTokenizer.from_pretrained(BASELINE_MODEL, revision=BASELINE_REVISION, trust_remote_code=False)
baseline = AutoModelForSequenceClassification.from_pretrained(BASELINE_MODEL,
    revision=BASELINE_REVISION, trust_remote_code=False, use_safetensors=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
baseline.to(device).eval()
baseline_probs = []
for start in range(0, len(test_frame), 32):
    inputs = baseline_tok(test_frame.text.iloc[start:start+32].tolist(), padding=True,
                          truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
    with torch.inference_mode():
        scores = baseline(**inputs).logits.softmax(-1).cpu().numpy()
    mapped = np.zeros((len(scores), 3))
    for i, label in baseline.config.id2label.items():
        mapped[:, LABEL_TO_ID[label.lower()]] = scores[:, int(i)]
    baseline_probs.append(mapped)
baseline_report = evaluation_report(test_frame, np.concatenate(baseline_probs))
(OUTPUT / "baseline_test_metrics.json").write_text(json.dumps(baseline_report, indent=2), encoding="utf-8")
print("Baseline and fine-tuned reports saved. Do not retune on this test set.")


## 7. Export privately and integrate after review
The export contains the full DistilBERT model, tokenizer, split ledger, training log and evaluation reports. The ledger and prediction files contain dataset membership information: keep the ZIP private. Do not commit model weights or raw data to the public GitHub repository.

After reviewing the reports, download the ZIP from Kaggle Output (or Colab Files). Extract `model` to your Windows project, e.g. `models/sentiment`. The existing Django adapter accepts named negative/neutral/positive labels and safetensors. In the same PowerShell window:

```powershell
$env:SENTIMENT_MODEL="C:\Users\jadaw\OneDrive\MINDBRIDGE\mindbridge\models\sentiment"
$env:SENTIMENT_REVISION="YOUR_EXPORTED_MODEL_SHA256"
$env:SENTIMENT_ENABLED="1"
$env:SENTIMENT_RISK_ENABLED="0"
.\.venv\Scripts\python.exe manage.py check_sentiment
```

Use the `artifact_sha256` value printed below as the local version identifier. For a local directory Transformers uses the files on disk; the revision value provides provenance to the Django records. Restart after changing settings. Keep the folder immutable after choosing its version.

A successful sentiment experiment still requires separate hybrid-alert evaluation (keywords + longitudinal mood + sentiment), including Tier-1 override and failed-model cases. Evaluate false negatives and false positives against reviewed risk labels; sentiment labels cannot establish alert accuracy.


In [ ]:
# Fingerprint the immutable deployable files, excluding reports that may be reviewed later.
artifact_hash = hashlib.sha256()
for path in sorted(MODEL_DIR.rglob("*")):
    if path.is_file():
        artifact_hash.update(str(path.relative_to(MODEL_DIR)).encode())
        with path.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                artifact_hash.update(chunk)
ARTIFACT_SHA256 = artifact_hash.hexdigest()
import importlib.metadata
metadata = {
    "status": "trained; evaluation review required before risk activation",
    "base_model": BASE_MODEL, "base_revision": BASE_REVISION,
    "artifact_sha256": ARTIFACT_SHA256, "input_csv_sha256": DATA_SHA256,
    "labels": LABELS, "seed": SEED, "max_length": MAX_LENGTH,
    "negative_threshold": THRESHOLD, "sentiment_risk_enabled": False,
    "versions": {name: importlib.metadata.version(name) for name in
                 ["torch", "transformers", "datasets", "accelerate", "scikit-learn"]},
    "python": platform.python_version(),
    "split_samples": df.split.value_counts().to_dict(),
    "split_groups": df.groupby("split").group_id.nunique().to_dict(),
    "limitations": ["English encoder; language-specific review required",
                    "Sentiment confidence is not clinical risk probability",
                    "No independent hybrid-alert accuracy claim"]}
(OUTPUT / "experiment.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
(OUTPUT / "training_log.json").write_text(json.dumps(trainer.state.log_history, indent=2), encoding="utf-8")
archive = OUTPUT.parent / "MindBridge_DistilBERT_Results.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUTPUT.rglob("*")):
        if path.is_file() and "checkpoints" not in path.relative_to(OUTPUT).parts:
            bundle.write(path, path.relative_to(OUTPUT))
print("Local model version:", ARTIFACT_SHA256)
print("Private result ZIP:", archive)


## References
- [DistilBERT base model card](https://huggingface.co/distilbert/distilbert-base-uncased)
- [Transformers 4.57 Trainer documentation](https://huggingface.co/docs/transformers/v4.57.0/en/main_classes/trainer)

Prepared for MindBridge. Pipeline checks can be validated before data is supplied; training outcomes must come from the actual reviewed data.
